In [1]:
# Imports and Setup
import os, ast
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import roc_auc_score, accuracy_score, precision_recall_fscore_support
from datasets import load_dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Load Model
try:
    model_name = "yuchuantian/AIGC_detector_env1"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
    model.eval()
    print(f"Loaded model: {model_name}")
except Exception as e:
    print(f"Error loading model: {e}")
    raise

# Scoring Function
def compute_aigc_score(text, model, tokenizer, device, max_len=512):

    try:
        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=max_len,
            padding=True
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits


        probs = torch.softmax(logits, dim=-1)
        ai_prob = probs[0][1].item()

        return ai_prob

    except Exception as e:
        print(f"Error scoring text: {e}")
        return 0.5

# Data Preparation
def prepare(df, col1, col2):

    X, y = [], []

    for _, r in df.iterrows():
        for col, lab in [(col1, 0), (col2, 1)]:
            val = r.get(col, None)

            if pd.isna(val):
                continue
            if isinstance(val, str) and val.strip().startswith('[{'):
                try:
                    parsed = ast.literal_eval(val)
                    for item in parsed:
                        if isinstance(item, dict):
                            for prompt_key in ['P1', 'P2', 'P3']:
                                if prompt_key in item:
                                    text = str(item[prompt_key]).strip()
                                    if text:
                                        X.append(text)
                                        y.append(lab)
                                    break
                except Exception as e:
                    text = str(val).strip()
                    if text:
                        X.append(text)
                        y.append(lab)
            else:
                text = str(val).strip()
                if text:
                    X.append(text)
                    y.append(lab)

    return X, y

# Evaluation Function
def evaluate_aigc(texts, labels, model, tokenizer, device):

    scores = []
    for t in tqdm(texts, desc="Scoring", ncols=80):
        s = compute_aigc_score(t, model, tokenizer, device)
        scores.append(s)

    try:
        auroc = roc_auc_score(labels, scores)
    except:
        auroc = 0.5
    thr = np.median(scores)
    preds = [1 if s > thr else 0 for s in scores]

    acc = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary', zero_division=0
    )

    return scores, preds, {
        'accuracy': acc,
        'auroc': auroc,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'threshold': thr
    }

# Load Dataset and Run Evaluation
print("\nLoading BEEMO dataset...")
df = load_dataset("toloka/beemo")['train'].to_pandas()
print(f"Loaded {len(df)} samples")

# Define tasks
tasks = {
    'H_vs_M': ('human_output', 'model_output'),
    'E_vs_M': ('human_edits', 'model_output'),
    'H_vs_E': ('human_output', 'human_edits'),
    'L_vs_M': ('llama-3.1-70b_edits', 'model_output'),
    'H_vs_L': ('human_output', 'llama-3.1-70b_edits'),
    'G_vs_M': ('gpt-4o_edits', 'model_output'),
    'H_vs_G': ('human_output', 'gpt-4o_edits'),
}

# Run tasks
PROJECT = "/content/drive/MyDrive/BEEMO_Reproducibility"
os.makedirs(f"{PROJECT}/results", exist_ok=True)
results = {}

for name, (c1, c2) in tasks.items():
    print(f"\n{'='*60}\n {name}\n{'='*60}")
    X, y = prepare(df, c1, c2)
    print(f"Samples: {len(X)} ({y.count(0)}/{y.count(1)})")

    scores, preds, metrics = evaluate_aigc(X, y, model, tokenizer, device)
    results[name] = metrics

    # Save detailed results
    pd.DataFrame({
        'text': X,
        'label': y,
        'score': scores,
        'pred': preds
    }).to_csv(
        f"{PROJECT}/results/{name}_aigc_mpu.csv",
        index=False
    )
    print(f"AUROC: {metrics['auroc']:.4f} | Acc: {metrics['accuracy']:.4f} | F1: {metrics['f1']:.4f}")

# Summary
summary = pd.DataFrame(results).T
summary.to_csv(f"{PROJECT}/results/aigc_mpu_summary.csv")

print("\n" + "="*60)
print("FINAL SUMMARY (AIGC MPU)")
print("="*60)
print(summary[['auroc', 'accuracy', 'f1']].round(4))
print(f"\n Results saved to {PROJECT}/results/")

Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

AUROC: 0.5730 | Acc: 0.5378 | F1: 0.6303

 G_vs_M
Samples: 8748 (6561/2187)


Scoring: 100%|██████████████████████████████| 8748/8748 [01:46<00:00, 82.01it/s]


AUROC: 0.5960 | Acc: 0.5449 | F1: 0.3932

 H_vs_G
Samples: 8748 (2187/6561)


Scoring: 100%|██████████████████████████████| 8748/8748 [01:39<00:00, 87.97it/s]


AUROC: 0.6277 | Acc: 0.5687 | F1: 0.6550

FINAL SUMMARY (AIGC MPU)
         auroc  accuracy      f1
H_vs_M  0.7053    0.6465  0.6465
E_vs_M  0.6726    0.6278  0.6278
H_vs_E  0.5411    0.5231  0.5231
L_vs_M  0.6562    0.5792  0.4390
H_vs_L  0.5730    0.5378  0.6303
G_vs_M  0.5960    0.5449  0.3932
H_vs_G  0.6277    0.5687  0.6550

 Results saved to /content/drive/MyDrive/BEEMO_Reproducibility/results/
